# ReliableAutoML vs. other AutoML frameworks (family-based comparison)

This notebook runs the comparison described in the paper *automl-comparison*
(`docs/latex/automl-comparison.tex`). Competitors are grouped by method family:

| Family | Frameworks | Role |
|---|---|---|
| (a) stacking / meta-ensemble | AutoGluon-Tabular, H2O AutoML, auto-sklearn | closest family to ReliableAutoML (meta-learned ensembles) |
| (b) cost-frugal search | FLAML | contrast: single tuned model, small budget |
| (c) evolutionary pipeline search | TPOT | contrast: genetic programming over sklearn pipelines |

**Outputs compared** (not just accuracy): selected model, held-out balanced accuracy / accuracy /
log-loss, wall-clock time, leaderboard size, and which reports/artefacts each framework returns.

**Protocol.** For each dataset $D$ and seed $s$ one stratified split
$D=D_{\text{dev}}\cup D_{\text{test}}$ (25% test, `random_state=s`) is drawn. Every framework sees
the same $D_{\text{dev}}$ and is scored once on the same $D_{\text{test}}$:

$$\mathrm{BA}=\frac1C\sum_{c=1}^{C}\frac{TP_c}{TP_c+FN_c},\qquad
\mathrm{LL}=-\frac1n\sum_{i=1}^{n}\log \hat p_{i,y_i}$$

(probabilities clipped to $[10^{-15},1]$ and renormalised). Competitors get a wall-clock budget
`BUDGET` seconds; ReliableAutoML has no time-budget parameter, so its measured wall time is
reported instead. Budgets, internal metrics and CV schemes differ between frameworks (see the
paper's feature table), so the numbers compare *delivered outputs under a stated budget*, not a
controlled ablation. Report mean ± std over seeds; with few datasets/seeds the result is
descriptive.

## 1. Parameters (edit this cell)

In [ ]:
# ---- PARAMETER CELL -------------------------------------------------------
# StatCore is not yet pushed/published, so the GitHub install below may fail or
# give an older ReliableML. If so set INSTALL_MODE = "zip" and upload a zip of
# the repository (containing pyproject.toml and the automl/ folder) in step 2b.
INSTALL_MODE = "git"            # "git" or "zip"
REPO_GIT = "git+https://github.com/Jabualani/ML-Reliabilty.git"
AUTOML_GIT = "git+https://github.com/Jabualani/ML-Reliabilty.git#subdirectory=automl"

FRAMEWORKS = ["reliableml", "sklearn_logistic", "autogluon", "h2o", "autosklearn",
              "flaml", "tpot"]
DATASETS = ["synthetic_mixed", "wine", "breast_cancer", "digits"]
SEEDS = [42, 43, 44]
BUDGET = 60                     # seconds per competitor fit
INSTALL_COMPETITORS = True
# ---------------------------------------------------------------------------

## 2a. Install ReliableML and ReliableML-AutoML

In [ ]:
import subprocess, sys
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    print(("OK   " if r.returncode == 0 else "FAIL ") + " ".join(args))
    if r.returncode != 0:
        print(r.stdout[-800:], r.stderr[-800:])
    return r.returncode == 0

if INSTALL_MODE == "git":
    pip(REPO_GIT)
    pip(AUTOML_GIT)
else:
    print("INSTALL_MODE='zip': run cell 2b")

## 2b. Fallback: upload a zip of the repository
Only used when `INSTALL_MODE = "zip"`.

In [ ]:
if INSTALL_MODE == "zip":
    import zipfile, pathlib
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    zipfile.ZipFile(name).extractall("repo_src")
    root = next(p.parent for p in pathlib.Path("repo_src").rglob("pyproject.toml")
                if p.parent.name != "automl")
    pip(str(root))
    pip(str(root / "automl"))

## 3. Install competitors (each independently; failures are recorded, not fatal)

auto-sklearn is Linux-only and its current releases require an older Python (3.10 or below);
a recent Colab runtime with Python 3.12 will usually fail to install it. In that case use a
Python-3.10 runtime or accept an `error` row for `autosklearn` - do not substitute numbers.
Installing several frameworks into one runtime can produce version conflicts; if a framework
fails, restart the runtime and install only that one.

In [ ]:
installed = {}
if INSTALL_COMPETITORS:
    pkgs = {"flaml": "flaml[automl]", "tpot": "tpot", "autogluon": "autogluon.tabular",
            "h2o": "h2o", "autosklearn": "auto-sklearn"}
    for fw, pkg in pkgs.items():
        if fw in FRAMEWORKS:
            installed[fw] = pip(pkg)
print(installed)

## 4. Experiment script (identical to `docs/latex/experiments/automl_comparison.py`)

In [ ]:
%%writefile automl_comparison.py
"""Family-based AutoML comparison on identical held-out splits.

Local use (only ReliableAutoML and a plain logistic baseline are run on the
development machine):
    .venv-automl\\Scripts\\python.exe docs\\latex\\experiments\\automl_comparison.py
        --frameworks reliableml sklearn_logistic --seeds 42 43 44
        --output docs\\latex\\experiments\\automl-comparison-results.json
Competitor adapters (flaml, tpot, autogluon, h2o, autosklearn) are only meant
for the Colab notebook docs/colab/automl.ipynb; each is wrapped so that an
installation or runtime failure is recorded instead of aborting the run.

Protocol: per (dataset, seed) one stratified 25% test split is made with
train_test_split(random_state=seed); every framework sees the same development
rows and is scored on the same untouched test rows. Competitors get a wall-clock
budget of --budget seconds; ReliableAutoML has no time budget and its measured
wall time is recorded.
"""
from __future__ import annotations

import argparse
import json
import platform
import shutil
import tempfile
import time
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.metrics import accuracy_score, balanced_accuracy_score, log_loss
from sklearn.model_selection import train_test_split

TARGET = "outcome"
TEST_FRACTION = 0.25


def make_mixed(seed: int, n: int = 240) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    age = rng.normal(42, 13, n)
    income = rng.lognormal(10.5, 0.55, n)
    region = rng.choice(["north", "south", "east", "west"], n)
    plan = rng.choice(["basic", "plus", "premium"], n, p=[0.5, 0.32, 0.18])
    score = (0.035 * (age - 40) + 0.7 * (plan == "premium")
             - 0.25 * (region == "south") + 0.35 * rng.normal(size=n))
    target = (score > np.quantile(score, 0.53)).astype(int)
    age[rng.random(n) < 0.08] = np.nan
    income[rng.random(n) < 0.07] = np.nan
    region = region.astype(object)
    plan = plan.astype(object)
    region[rng.random(n) < 0.06] = None
    plan[rng.random(n) < 0.05] = None
    return pd.DataFrame({"age": age, "income": income, "region": region,
                         "plan": plan, TARGET: target})


def load_frame(name: str, seed: int) -> pd.DataFrame:
    if name == "synthetic_mixed":
        return make_mixed(seed)
    loader = {"wine": load_wine, "breast_cancer": load_breast_cancer,
              "digits": load_digits}[name]
    bunch = loader()
    frame = pd.DataFrame(bunch.data, columns=[str(c) for c in bunch.feature_names])
    frame[TARGET] = bunch.target
    return frame


def split(frame: pd.DataFrame, seed: int):
    return train_test_split(frame, test_size=TEST_FRACTION, random_state=seed,
                            stratify=frame[TARGET])


def metric_record(y_true, prediction, probability, classes) -> dict:
    probability = np.clip(np.asarray(probability, dtype=float), 1e-15, 1.0)
    probability = probability / probability.sum(axis=1, keepdims=True)
    return {"accuracy": float(accuracy_score(y_true, prediction)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, prediction)),
            "log_loss": float(log_loss(y_true, probability, labels=list(classes)))}


def numeric_design(train_X: pd.DataFrame):
    """Median-impute + one-hot transformer for frameworks needing numeric input."""
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
    numeric = train_X.select_dtypes(include=[np.number]).columns.tolist()
    categorical = [c for c in train_X.columns if c not in numeric]
    parts = []
    if numeric:
        parts.append(("num", SimpleImputer(strategy="median"), numeric))
    if categorical:
        parts.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
            categorical))
    return ColumnTransformer(parts)


def describe_object(obj) -> dict:
    """Describe the structure of a report object without assuming its class."""
    info = {"type": type(obj).__name__}
    if isinstance(obj, dict):
        info["keys"] = sorted(str(k) for k in obj)
    elif hasattr(obj, "__dict__"):
        info["fields"] = sorted(k for k in vars(obj) if not k.startswith("_"))
    return info


# --------------------------------------------------------------------------
# Adapters: each returns (prediction, probability, classes, info)
# --------------------------------------------------------------------------
def run_reliableml(train, test_X, seed, budget):
    from reliableml_automl import ReliableAutoML
    automl = ReliableAutoML(target_column=TARGET, random_state=seed,
                            partitioner="s_attp_clf", include_statcore=True,
                            tuning="light", cv=3, tuning_cv=2,
                            progress_bar=False).fit(train)
    prediction = np.asarray(automl.predict(test_X))
    probability = np.asarray(automl.best_pipeline.predict_proba(test_X))
    classes = list(automl.best_pipeline.classes_)
    fit_report = automl.fit_report
    info = {"selected_model": automl.selected_model_name,
            "leaderboard_size": len(automl.leaderboard),
            "leaderboard": automl.leaderboard,
            "partition_sizes": fit_report.get("partition_sizes"),
            "n_warnings": len(fit_report.get("warnings", [])),
            "cleaning_report_keys": sorted(automl.cleaning_report),
            "fit_report_keys": sorted(fit_report)}
    try:
        _, report = automl.predict_with_report(test_X)
        info["predict_with_report"] = describe_object(report)
    except Exception as error:  # recorded, not fatal
        info["predict_with_report"] = {"error": repr(error)}
    directory = Path(tempfile.mkdtemp(prefix="raml_save_", dir=Path.cwd()))
    try:
        automl.save(directory / "artifact")
        files = [p for p in directory.rglob("*") if p.is_file()]
        info["saved_artifact_files"] = len(files)
        info["saved_artifact_bytes"] = int(sum(p.stat().st_size for p in files))
    except Exception as error:
        info["saved_artifact"] = {"error": repr(error)}
    finally:
        shutil.rmtree(directory, ignore_errors=True)
    return prediction, probability, classes, info


def run_sklearn_logistic(train, test_X, seed, budget):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    X, y = train.drop(columns=[TARGET]), train[TARGET]
    pipe = Pipeline([("prep", numeric_design(X)), ("scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=3000,
                                                class_weight="balanced"))]).fit(X, y)
    return (pipe.predict(test_X), pipe.predict_proba(test_X),
            list(pipe.classes_), {"selected_model": "LogisticRegression"})


def run_flaml(train, test_X, seed, budget):
    from flaml import AutoML
    X, y = train.drop(columns=[TARGET]), train[TARGET]
    automl = AutoML()
    automl.fit(X, y, task="classification", time_budget=budget, seed=seed,
               verbose=0, n_jobs=1)
    return (automl.predict(test_X), automl.predict_proba(test_X),
            list(automl.classes_),
            {"selected_model": str(automl.best_estimator),
             "best_config": str(automl.best_config),
             "optimised_metric": "flaml default (log_loss for classification)"})


def run_tpot(train, test_X, seed, budget):
    from tpot import TPOTClassifier
    X, y = train.drop(columns=[TARGET]), train[TARGET]
    design = numeric_design(X).fit(X)
    tpot = TPOTClassifier(search_space="linear-light",
                          scorers=["balanced_accuracy"], scorers_weights=[1],
                          cv=3, max_time_mins=budget / 60.0, random_state=seed,
                          n_jobs=1, verbose=0)
    tpot.fit(design.transform(X), y)
    model = tpot.fitted_pipeline_
    Xt = design.transform(test_X)
    return (model.predict(Xt), model.predict_proba(Xt), list(model.classes_),
            {"selected_model": str(model)[:300]})


def run_autogluon(train, test_X, seed, budget):
    from autogluon.tabular import TabularPredictor
    directory = Path(tempfile.mkdtemp(prefix="ag_", dir=Path.cwd()))
    try:
        predictor = TabularPredictor(label=TARGET, eval_metric="balanced_accuracy",
                                     path=str(directory), verbosity=0)
        predictor.fit(train, time_limit=budget, presets="best_quality")
        board = predictor.leaderboard(silent=True)
        proba = predictor.predict_proba(test_X)
        info = {"selected_model": str(predictor.model_best),
                "leaderboard_size": int(len(board)),
                "preset": "best_quality"}
        return (predictor.predict(test_X).to_numpy(), proba.to_numpy(),
                list(proba.columns), info)
    finally:
        shutil.rmtree(directory, ignore_errors=True)


def run_h2o(train, test_X, seed, budget):
    import h2o
    from h2o.automl import H2OAutoML
    h2o.init(verbose=False, nthreads=1)
    h2o.no_progress()
    try:
        frame = train.copy()
        frame[TARGET] = frame[TARGET].astype(str)
        hf = h2o.H2OFrame(frame)
        hf[TARGET] = hf[TARGET].asfactor()
        aml = H2OAutoML(max_runtime_secs=budget, seed=seed, nfolds=5,
                        sort_metric="mean_per_class_error", verbosity=None)
        aml.train(y=TARGET, training_frame=hf)
        out = aml.leader.predict(h2o.H2OFrame(test_X)).as_data_frame()
        classes = [c for c in out.columns if c != "predict"]
        order = sorted(classes)
        return (out["predict"].to_numpy(), out[order].to_numpy(), order,
                {"selected_model": str(aml.leader.model_id),
                 "leaderboard_size": int(aml.leaderboard.nrows)})
    finally:
        h2o.cluster().shutdown(prompt=False)


def run_autosklearn(train, test_X, seed, budget):
    import autosklearn.classification as ask
    X, y = train.drop(columns=[TARGET]), train[TARGET]
    for column in X.columns:
        if X[column].dtype == object:
            X[column] = X[column].astype("category")
            test_X[column] = pd.Categorical(test_X[column],
                                            categories=X[column].cat.categories)
    model = ask.AutoSklearnClassifier(time_left_for_this_task=int(budget),
                                      seed=seed, n_jobs=1,
                                      metric=__import__("autosklearn.metrics",
                                                        fromlist=["x"]).balanced_accuracy)
    model.fit(X, y)
    return (model.predict(test_X), model.predict_proba(test_X),
            list(model.classes_),
            {"selected_model": f"ensemble of {len(model.get_models_with_weights())} models"})


ADAPTERS = {"reliableml": run_reliableml, "sklearn_logistic": run_sklearn_logistic,
            "flaml": run_flaml, "tpot": run_tpot, "autogluon": run_autogluon,
            "h2o": run_h2o, "autosklearn": run_autosklearn}


def versions() -> dict:
    out = {}
    for name in ("numpy", "pandas", "sklearn", "flaml", "tpot", "autogluon.tabular",
                 "h2o", "autosklearn", "reliableml_automl", "ml_reliability"):
        try:
            module = __import__(name, fromlist=["x"])
            out[name] = getattr(module, "__version__", "unknown")
        except Exception:
            out[name] = None
    return out


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--frameworks", nargs="+", default=["reliableml", "sklearn_logistic"])
    parser.add_argument("--datasets", nargs="+",
                        default=["synthetic_mixed", "wine", "breast_cancer", "digits"])
    parser.add_argument("--seeds", type=int, nargs="+", default=[42, 43, 44])
    parser.add_argument("--budget", type=float, default=60.0,
                        help="seconds per competitor fit (ReliableAutoML ignores it)")
    parser.add_argument("--output", default=None)
    args = parser.parse_args()
    warnings.filterwarnings("ignore")
    meta = {"python": platform.python_version(), "platform": platform.platform(),
            "versions": versions(), "seeds": args.seeds, "datasets": args.datasets,
            "test_fraction": TEST_FRACTION, "budget_seconds": args.budget,
            "frameworks": args.frameworks}
    records = []
    for seed in args.seeds:
        for dataset in args.datasets:
            frame = load_frame(dataset, seed)
            train, test = split(frame, seed)
            test_X, y_test = test.drop(columns=[TARGET]), test[TARGET].to_numpy()
            for name in args.frameworks:
                record = {"framework": name, "dataset": dataset, "seed": seed,
                          "n_development": len(train), "n_test": len(test),
                          "n_features": frame.shape[1] - 1,
                          "n_classes": int(frame[TARGET].nunique())}
                start = time.perf_counter()
                try:
                    pred, proba, classes, info = ADAPTERS[name](
                        train.copy(), test_X.copy(), seed, args.budget)
                    record["wall_seconds"] = time.perf_counter() - start
                    record.update(metric_record(y_test, pred, proba, classes))
                    record["info"] = info
                except Exception as error:
                    record["wall_seconds"] = time.perf_counter() - start
                    record["error"] = repr(error)
                    record["traceback_tail"] = traceback.format_exc()[-600:]
                records.append(record)
                shown = {k: record.get(k) for k in
                         ("framework", "dataset", "seed", "balanced_accuracy",
                          "log_loss", "wall_seconds", "error")}
                print(json.dumps(shown, default=str), flush=True)
    if args.output:
        Path(args.output).write_text(
            json.dumps({"metadata": meta, "records": records}, indent=1, default=str),
            encoding="utf-8")


if __name__ == "__main__":
    main()


## 5. Run

Each framework runs in its own subprocess so a crash in one cannot abort the others.
Expected runtime is roughly `len(DATASETS) * len(SEEDS) * BUDGET` seconds per competitor.

In [ ]:
import json, subprocess, sys, pathlib
parts = []
for fw in FRAMEWORKS:
    out = f"results_{fw}.json"
    cmd = [sys.executable, "automl_comparison.py", "--frameworks", fw,
           "--datasets", *DATASETS, "--seeds", *map(str, SEEDS),
           "--budget", str(BUDGET), "--output", out]
    print(">>", fw, flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-1500:]); 
    if r.returncode != 0:
        print(r.stderr[-1500:])
    if pathlib.Path(out).exists():
        parts.append(json.loads(pathlib.Path(out).read_text()))
records = [rec for p in parts for rec in p["records"]]
metadata = parts[0]["metadata"] if parts else {}
pathlib.Path("automl_comparison_results.json").write_text(
    json.dumps({"metadata": metadata, "records": records}, indent=1))
print(len(records), "records")

## 6. Results table (paste this back into the paper)

Columns match the paper tables: balanced accuracy, accuracy, log-loss, wall seconds (mean ± std
over seeds) and the number of successful / failed runs. Failed runs are listed separately and are
excluded from the mean.

In [ ]:
import pandas as pd, numpy as np
df = pd.DataFrame(records)
if "error" not in df: df["error"] = np.nan
ok = df[df["error"].isna()]
def ms(x): return f"{x.mean():.4f} ± {x.std(ddof=1) if len(x) > 1 else float('nan'):.4f}"
rows = []
for (ds, fw), g in ok.groupby(["dataset", "framework"]):
    rows.append({"dataset": ds, "framework": fw, "n_ok": len(g),
                 "n_failed": int(((df.dataset == ds) & (df.framework == fw) & df.error.notna()).sum()),
                 "balanced_accuracy": ms(g.balanced_accuracy), "accuracy": ms(g.accuracy),
                 "log_loss": ms(g.log_loss), "wall_seconds": f"{g.wall_seconds.mean():.1f} ± {g.wall_seconds.std(ddof=1) if len(g)>1 else float('nan'):.1f}"})
table = pd.DataFrame(rows).sort_values(["dataset", "framework"])
table.to_csv("automl_comparison_table.csv", index=False)
print(table.to_string(index=False))
print("\nFAILED RUNS")
print(df[df.error.notna()][["framework", "dataset", "seed", "error"]].drop_duplicates(["framework", "error"]).to_string(index=False)
      if df.error.notna().any() else "none")

## 7. Selected models and reports returned (qualitative outputs)

In [ ]:
for fw, g in ok.groupby("framework"):
    sel = g["info"].apply(lambda d: d.get("selected_model") if isinstance(d, dict) else None)
    print(f"{fw:18s}", sorted(set(map(str, sel)))[:6])
raml = [r for r in records if r["framework"] == "reliableml" and "info" in r]
if raml:
    info = raml[0]["info"]
    print("\nReliableAutoML extra outputs:", {k: info[k] for k in info if k != "leaderboard"})

## 8. Download

Download `automl_comparison_results.json` and `automl_comparison_table.csv` and paste the table
into the *to be filled from the Colab run* cells of `automl-comparison.tex`.

In [ ]:
from google.colab import files
files.download("automl_comparison_results.json")
files.download("automl_comparison_table.csv")